In [26]:
import pandas as pd
import geopandas as gpd
import json
from shapely.geometry import shape
from shapely import wkt
import warnings
# filter all warnings
warnings.filterwarnings("ignore")

In [28]:
with open('Data/Metro/metro_stops.json') as f:
    data = json.load(f)

df_metro_stops = []
for stop in data['features']:
    properties = stop.get('properties', {})
    nom_estacio = properties.get('NOM_ESTACIO')
    linia = properties.get('PICTO')
    id = properties.get('ID_ESTACIO')
    df_metro_stops.append({
            'id': id,
            'name': nom_estacio,
            'linia': linia,
            'stop_type': 'Metro',
            'geometry': shape(stop['geometry'])
        })


geo_df_metro_stops = gpd.GeoDataFrame(df_metro_stops, geometry='geometry')
excluded = ['FM','TM']
geo_df_metro_stops = geo_df_metro_stops[~geo_df_metro_stops['linia'].isin(excluded)]
geo_df_metro_stops

,id,name,linia,stop_type,geometry
0,111,Hospital de Bellvitge,L1,Metro,POINT (2.10724 41.34468)
1,112,Bellvitge,L1,Metro,POINT (2.11092 41.35097)
2,113,Av. Carrilet,L1,Metro,POINT (2.10265 41.35852)
3,114,Rambla Just Oliveras,L1,Metro,POINT (2.09975 41.36409)
4,115,Can Serra,L1,Metro,POINT (2.10275 41.36769)
...,...,...,...,...,...
133,959,Provençana,L10S,Metro,POINT (2.1241 41.36135)
134,1137,Casa de l'Aigua,L11,Metro,POINT (2.18484 41.45128)
135,1138,Torre Baró | Vallbona,L11,Metro,POINT (2.17988 41.4592)
136,1139,Ciutat Meridiana,L11,Metro,POINT (2.17465 41.46081)


# Solution nodes

Theya are the same as the original stops

fara falta un filtrat de parades les quals considerem possible soslucions

In [ ]:
solution_nodes = geo_df_metro_stops.copy()
solution_nodes = solution_nodes.drop(columns=['linia'])
solution_nodes['id'] = 'SM' + '-' + solution_nodes['id'].astype(str)

,id,name,stop_type,geometry
0,SM-111,Hospital de Bellvitge,Metro,POINT (2.10724 41.34468)
1,SM-112,Bellvitge,Metro,POINT (2.11092 41.35097)
2,SM-113,Av. Carrilet,Metro,POINT (2.10265 41.35852)
3,SM-114,Rambla Just Oliveras,Metro,POINT (2.09975 41.36409)
4,SM-115,Can Serra,Metro,POINT (2.10275 41.36769)
...,...,...,...,...
133,SM-959,Provençana,Metro,POINT (2.1241 41.36135)
134,SM-1137,Casa de l'Aigua,Metro,POINT (2.18484 41.45128)
135,SM-1138,Torre Baró | Vallbona,Metro,POINT (2.17988 41.4592)
136,SM-1139,Ciutat Meridiana,Metro,POINT (2.17465 41.46081)


# Metro Nodes

En fem un per cada Línia que tinguin

In [30]:
geo_df_metro_stops["linia"] = geo_df_metro_stops["linia"].str.findall(r"L\d+")
geo_df_metro_stops = geo_df_metro_stops.explode("linia", ignore_index=True)
geo_df_metro_stops


,id,name,linia,stop_type,geometry
0,111,Hospital de Bellvitge,L1,Metro,POINT (2.10724 41.34468)
1,112,Bellvitge,L1,Metro,POINT (2.11092 41.35097)
2,113,Av. Carrilet,L1,Metro,POINT (2.10265 41.35852)
3,114,Rambla Just Oliveras,L1,Metro,POINT (2.09975 41.36409)
4,115,Can Serra,L1,Metro,POINT (2.10275 41.36769)
...,...,...,...,...,...
164,959,Provençana,L10,Metro,POINT (2.1241 41.36135)
165,1137,Casa de l'Aigua,L11,Metro,POINT (2.18484 41.45128)
166,1138,Torre Baró | Vallbona,L11,Metro,POINT (2.17988 41.4592)
167,1139,Ciutat Meridiana,L11,Metro,POINT (2.17465 41.46081)


## Afegir les parades de metro amb accés a fgc

In [31]:
pl_cat_l6 = geo_df_metro_stops[geo_df_metro_stops['name'] == 'Catalunya'].drop_duplicates(subset=['id'], keep='first')
pl_cat_l6['linia'] = 'L6'
pl_cat_l7 = pl_cat_l6.copy()
pl_cat_l7['linia'] = 'L7'

diagonal_l6 = geo_df_metro_stops[geo_df_metro_stops['name'] == 'Diagonal'].drop_duplicates(subset=['id'], keep='first')
diagonal_l6['linia'] = 'L6'
diagonal_l7 = diagonal_l6.copy()
diagonal_l7['linia'] = 'L7'

carrilet_l8 = geo_df_metro_stops[geo_df_metro_stops['name'] == 'Av. Carrilet']
carrilet_l8['linia'] = 'L8'

eu_i_fira_l8 = geo_df_metro_stops[geo_df_metro_stops['name'] == 'Europa | Fira']
eu_i_fira_l8['linia'] = 'L8'

geo_df_metro_stops = pd.concat([geo_df_metro_stops, pl_cat_l6, pl_cat_l7, diagonal_l6, diagonal_l7, carrilet_l8, eu_i_fira_l8], ignore_index=True)


In [32]:
geo_df_metro_stops['id'] = 'M' + '-' + geo_df_metro_stops['linia'] + '-' + geo_df_metro_stops['id'].astype(str)
geo_df_metro_stops

,id,name,linia,stop_type,geometry
0,M-L1-111,Hospital de Bellvitge,L1,Metro,POINT (2.10724 41.34468)
1,M-L1-112,Bellvitge,L1,Metro,POINT (2.11092 41.35097)
2,M-L1-113,Av. Carrilet,L1,Metro,POINT (2.10265 41.35852)
3,M-L1-114,Rambla Just Oliveras,L1,Metro,POINT (2.09975 41.36409)
4,M-L1-115,Can Serra,L1,Metro,POINT (2.10275 41.36769)
...,...,...,...,...,...
170,M-L7-126,Catalunya,L7,Metro,POINT (2.16972 41.38771)
171,M-L6-328,Diagonal,L6,Metro,POINT (2.16044 41.39568)
172,M-L7-328,Diagonal,L7,Metro,POINT (2.16044 41.39568)
173,M-L8-113,Av. Carrilet,L8,Metro,POINT (2.10265 41.35852)
